# Build from RTL vs. use the packaged bitstream

`spikeengine` ships pre-built, timing-closed bitstreams for all three supported FPGA families (Basys3, SP701, ZCU104) -- the fast path, `superneuromat.spikeengine.program.program(board)`, loads one of these in seconds. But the full RTL sources, Vivado constraints, and build scripts are ALSO bundled in the package, so you can instead run a real synth -> place -> route -> write_bitstream flow yourself and program the FRESH result (`superneuromat.spikeengine.build.build_bitstream(board)`) -- e.g. after modifying the RTL, or just to reproduce the packaged build from source.

This notebook shows both paths for each of the three boards, and reports real Vivado timing/resource numbers for the from-RTL builds (not just claims).

In [ ]:
from superneuromat import spikeengine as se
from superneuromat.spikeengine import build as se_build
from superneuromat.spikeengine import program as se_program
from superneuromat.spikeengine.boards import get_board, list_boards

## Configuration

`SOURCE = 'packaged'` uses the shipped bitstream directly (seconds). `SOURCE = 'build'` runs a REAL Vivado build from RTL first (10-40+ minutes per board, real synthesis -- this is the expensive, opt-in path, never triggered silently). `BOARDS` controls which of the three families to cover in this run.

In [ ]:
SOURCE = 'build'         # 'packaged' (fast) or 'build' (real Vivado synth, slow)
BOARDS = list_boards()   # ['basys3', 'sp701', 'zcu104'] -- or a subset, e.g. ['basys3']
PROGRAM_HARDWARE = False  # actually JTAG-program a connected board after resolving the bitstream
PORT = 'auto'

## Resolve a bitstream per board

For `SOURCE='packaged'`, this is just `get_board(board).bitstream_path()` -- the file that ships with the package, already timing-closed and (for basys3) hardware-validated. For `SOURCE='build'`, `build_bitstream()` runs the board's bundled Tcl build script through Vivado batch mode and returns a NEW bitstream plus real post-route numbers -- it does NOT touch or overwrite the packaged one, so both remain available to compare.

In [ ]:
results = {}
for board in BOARDS:
    print(f'--- {board} ({SOURCE}) ---')
    if SOURCE == 'packaged':
        spec = get_board(board)
        bit = spec.bitstream_path()
        print(f'  packaged bitstream: {bit}')
        print(f'  exists: {bit.exists()}')
        print(f'  RTL sources:        {se_build.rtl_source_dir()}')
        results[board] = {'source': 'packaged', 'bitstream_path': bit, 'outdir': None,
                              'rtl_dir': se_build.rtl_source_dir(), 'wns_ns': None,
                              'lut': None, 'ff': None, 'bram': None, 'uram': None,
                              'drc_errors': None, 'drc_warnings': None,
                              'hardware_validated': spec.hardware_validated}
    elif SOURCE == 'build':
        r = se_build.build_bitstream(board)   # real Vivado run -- see build.py's docstring
        print(f'  built bitstream:    {r.bitstream_path}')
        print(f'  WNS={r.wns_ns} ns  LUT={r.lut}  FF={r.ff}  BRAM={r.bram}  URAM={r.uram}  '
              f'DRC errors={r.drc_errors} warnings={r.drc_warnings}')
        # outdir holds EVERY artifact this build generated, not just the .bit --
        # checkpoints (post_synth.dcp/post_route.dcp) and the full report set
        # (timing/utilization/DRC/power/worst-paths). rtl_dir is the source RTL
        # this build actually synthesized -- both stay on disk, nothing hidden.
        print(f'  build outdir:       {r.outdir}')
        print(f'  RTL sources used:   {r.rtl_dir}')
        results[board] = {'source': 'build', 'bitstream_path': r.bitstream_path, 'outdir': r.outdir,
                              'rtl_dir': r.rtl_dir, 'wns_ns': r.wns_ns,
                              'lut': r.lut, 'ff': r.ff, 'bram': r.bram, 'uram': r.uram,
                              'drc_errors': r.drc_errors, 'drc_warnings': r.drc_warnings,
                              'hardware_validated': False}   # a fresh build is unverified until tested
    else:
        raise ValueError(f"SOURCE must be 'packaged' or 'build', got {SOURCE!r}")

## Summary

In [ ]:
print(f'{"board":8s} {"source":9s} {"WNS(ns)":>8s} {"LUT":>7s} {"FF":>7s} {"BRAM":>6s} {"URAM":>6s} {"DRC-err":>8s}')
for board, r in results.items():
    wns = f'{r["wns_ns"]:.3f}' if r['wns_ns'] is not None else '-'
    print(f'{board:8s} {r["source"]:9s} {wns:>8s} {r["lut"] or "-"!s:>7s} '
          f'{r["ff"] or "-"!s:>7s} {r["bram"] or "-"!s:>6s} {r["uram"] or "-"!s:>6s} '
          f'{r["drc_errors"] if r["drc_errors"] is not None else "-"!s:>8s}')

## (Optional) Program the resolved bitstream

Set `PROGRAM_HARDWARE = True` above (and pick a single connected board in `BOARDS`) to actually JTAG-program it -- volatile load only (lost on power-cycle, flash untouched).

In [ ]:
from superneuromat import SNN

LEAK = 1000.0   # see logic_gates.ipynb -- stands in for SuperNeuroMAT's default leak=inf,
                 # which the board's fixed-point config registers can't represent

def _or_gate_check(dev):
    """A real functional test, not just a protocol health check: builds the
    tiny OR-gate network from logic_gates.ipynb and verifies all 4 truth-table
    rows on-chip. Proves the resolved bitstream actually computes correctly,
    not just that it responds to READ_STATUS/soft_reset."""
    net = SNN()
    ins = [net.create_neuron(threshold=0, leak=LEAK).idx for _ in range(2)]
    out = net.create_neuron(threshold=0, leak=LEAK).idx
    for i in ins:
        net.create_synapse(i, out, weight=1)
    n_neurons = len(net.neuron_thresholds)
    ok = True
    for x in (0, 1):
        for y in (0, 1):
            dev.soft_reset()
            se.load_network(dev, net, frac_bits=0)
            spikes = se.run_schedule(dev, {0: {ins[0]: x, ins[1]: y}}, total_ticks=2,
                                     frac_bits=0, n_neurons=n_neurons)
            got, expect = bool(spikes[-1][out]), bool(x or y)
            ok = ok and (got == expect)
    return ok


if PROGRAM_HARDWARE:
    for board, r in results.items():
        print(f'programming {board} with {r["bitstream_path"]} ...')
        se_program.program(board, bitstream=r['bitstream_path'])
        dev = se.connect(port=PORT, board=board)
        dev.clear_error()
        status = dev.read_status()
        dev.soft_reset()   # requires OP_ENGINE_RESET -- proves this is really the STDP bitstream
        print(f'  {board}: read_status={status}, soft_reset OK')
        functional_ok = _or_gate_check(dev)
        print(f'  {board}: OR-gate functional check (all 4 truth-table rows): '
              f'{"PASS" if functional_ok else "FAIL"}')
        dev.close()
else:
    print('PROGRAM_HARDWARE is False -- not touching any board. Set it True to program + smoke-test.')